# 📘 Session 22: Unsupervised Learning and Clustering
### Duration: ~2.5–3 Hours

---

**Topics Covered:**
1. Supervised vs Unsupervised Learning — Recap
2. What Is Clustering? — Goals, Distance, and When to Use It
3. A Toy Example — `make_blobs` and Visual Intuition
4. K-Means Clustering — Algorithm and scikit-learn API
5. The Wine Dataset — Features for Clustering
6. Preprocessing for Clustering — Why Scaling Matters
7. Choosing the Number of Clusters — Elbow Method and Silhouette Score
8. Evaluating and Interpreting Clusters
9. Visualizing Clusters — 2D Scatter and PCA
10. Density-Based Clustering — `DBSCAN` (Brief)
11. Clustering Workflow and Real-World Use Cases

---

**Why This Session Matters for Data Science:**
- Many datasets have **no labels** — customer segments, product groups, operational cohorts
- **Clustering** discovers structure you can explore, report, or feed into later supervised models
- **Scaling** strongly affects distance-based methods like K-Means
- We use the **Wine** dataset — a fresh example, **not** Palmer Penguins (our supervised thread in Sessions 15–21)

> **Prerequisites:** Sessions 16–21 (preprocessing, supervised models, metrics). Session 16 introduced unsupervised learning in one table row; today we go deep on **clustering**.

> **Not in this session:** PCA as dimensionality reduction for modeling, topic modeling (LDA), neural embeddings, hierarchical clustering dendrograms in detail.

**Libraries:** `pandas`, `numpy`, `matplotlib`, **`scikit-learn`**



---
## 1. Supervised vs Unsupervised Learning — Recap

| Type | Labels (y)? | Goal | Sessions so far |
|------|-------------|------|-----------------|
| **Supervised** | Yes | Predict target | 17–21 (penguin species, house prices) |
| **Unsupervised** | No | Find patterns / groups | **Today** |

### Common unsupervised tasks

| Task | Question | Example |
|------|----------|---------|
| **Clustering** | Which rows are similar? | Wine styles from chemistry |
| **Dimensionality reduction** | Can we summarize many columns? | PCA (preview only today) |
| **Anomaly detection** | Which points are unusual? | Fraud outliers |

### Clustering vs classification

| | Classification (supervised) | Clustering (unsupervised) |
|--|----------------------------|---------------------------|
| Training | Uses known class labels | **No** label column in `fit` |
| Output | Predict class for new rows | Assign cluster IDs (0, 1, 2, …) |
| Success metric | Accuracy, F1 on test labels | Silhouette, cluster profiles, business fit |

> ⚠️ **Special Case — hidden labels for teaching only**: Wine has cultivar classes in sklearn, but we **do not pass them to** `fit`. We compare clusters to cultivar **afterward** only to interpret results. In real projects you may never have ground truth.

> **Data Science relevance**: Clustering is exploratory — name segments from **profiles**, not just from metrics.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score



---
## 2. What Is Clustering?

**Clustering** groups rows so that:
- Points **inside** a cluster are similar
- Points in **different** clusters are less similar

### K-Means (today's main algorithm)

1. Pick **K** (number of clusters)
2. Place **K centroids** (cluster centers)
3. Assign each point to the nearest centroid
4. Move centroids to the mean of assigned points
5. Repeat until assignments stabilize

| Hyperparameter | Meaning |
|----------------|---------|
| `n_clusters` (K) | How many groups to form |
| `random_state` | Reproducible initialization |
| `n_init` | Number of runs with different seeds |

> **Distance**: K-Means uses **Euclidean distance** — features on large scales (e.g. `proline`) dominate unless you **scale**.



---
## 3. Toy Example — `make_blobs`

Synthetic 2D data with known groups builds intuition before real wine chemistry data.



In [ ]:
X_blob, y_blob = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.9,
    random_state=42,
)

plt.figure(figsize=(7, 5))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("make_blobs — True Groups (for intuition)")
plt.colorbar(label="blob label")
plt.tight_layout()
plt.show()

km_blob = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_blob = km_blob.fit_predict(X_blob)

plt.figure(figsize=(7, 5))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=labels_blob, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.scatter(
    km_blob.cluster_centers_[:, 0],
    km_blob.cluster_centers_[:, 1],
    c="red", marker="X", s=200, linewidths=2, label="centroids",
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("K-Means (K=3) on Blobs")
plt.legend()
plt.tight_layout()
plt.show()

print("Silhouette score (blobs):", round(silhouette_score(X_blob, labels_blob), 3))



---
## 4. K-Means API — `fit`, `fit_predict`, `cluster_centers_`

| Method / attribute | Purpose |
|--------------------|---------|
| `.fit(X)` | Learn centroids from features only |
| `.fit_predict(X)` | Fit and return cluster label per row |
| `.predict(X)` | Assign clusters for new data (after fit) |
| `.cluster_centers_` | Coordinates of centroids (in scaled space if you scaled first) |
| `.inertia_` | Sum of squared distances to nearest centroid (lower = tighter) |

**No `y`** is passed — clustering is unsupervised.



---
## 5. The Wine Dataset — Our Clustering Project

**Wine** = 178 Italian wine samples, **13 numeric chemistry features** (alcohol, acids, phenols, color, `proline`, etc.).

| Item | Value |
|------|-------|
| **Rows** | 178 wines |
| **Features (X)** | 13 lab measurements (all numeric) |
| **Hidden label** | 3 cultivar classes (`class_0`, `class_1`, `class_2`) — for validation only |

**Story today:** A winery analyst groups wines by **chemical profile** — no cultivar column during clustering.



In [ ]:
wine_bundle = load_wine(as_frame=True)
wine_df = wine_bundle.frame.copy()

cultivar_true = wine_bundle.target
cultivar_names = wine_bundle.target_names

feature_cols = list(wine_bundle.feature_names)
X_wine = wine_df[feature_cols].copy()

print("Shape:", X_wine.shape)
print("Cultivars (not used in fit):", list(cultivar_names))
print()
print(X_wine.describe().round(2))



---
## 6. Preprocessing for Clustering — Scale, Then K-Means

Compare scales: `proline` ranges ~278–1680 while `hue` is ~0.5–1.7. **StandardScaler** puts each feature on comparable scale before distance-based clustering.



In [ ]:
scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)

km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = km3.fit_predict(X_wine_scaled)

wine_result = wine_df.copy()
wine_result["cluster"] = cluster_labels

print("Cluster counts:\n", wine_result["cluster"].value_counts().sort_index())
print("\nCluster centers (original scale):")
centers_orig = scaler.inverse_transform(km3.cluster_centers_)
centers_df = pd.DataFrame(centers_orig, columns=feature_cols)
print(centers_df.round(2).to_string())



---
## 7. Choosing K — Elbow Method and Silhouette

In production you often **do not know** K in advance. Try several values and compare:

| Method | Idea |
|--------|------|
| **Elbow** | Plot `inertia_` vs K; look for a bend |
| **Silhouette** | Score in [-1, 1]; higher = better separation |

> Silhouette uses cluster geometry — it does **not** need true labels.



In [ ]:
K_range = range(2, 9)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_wine_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_wine_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), inertias, "o-")
axes[0].set_xlabel("K (n_clusters)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Plot — Wine")
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), sil_scores, "s-", color="forestgreen")
axes[1].set_xlabel("K (n_clusters)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette vs K")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(sil_scores))]
print("Best K by silhouette:", best_k, "| score:", round(max(sil_scores), 3))



---
## 8. Evaluating Clusters — Profiles and Hidden Cultivar

### Cluster profiles (what practitioners actually read)

Mean chemistry per cluster helps **name** segments (e.g. high-alcohol, high-proline group).

### Optional validation metrics (labels held out during fit)

| Metric | Needs true labels? | Meaning |
|--------|-------------------|---------|
| **Silhouette** | No | Separation quality |
| **Adjusted Rand Index (ARI)** | Yes | Agreement with cultivar (1 = perfect, ~0 = random) |



In [ ]:
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_best = km_best.fit_predict(X_wine_scaled)

print("Silhouette (K=%d):" % best_k, round(silhouette_score(X_wine_scaled, labels_best), 3))
print("Adjusted Rand Index vs cultivar:", round(adjusted_rand_score(cultivar_true, labels_best), 3))

profile_cols = ["alcohol", "flavanoids", "color_intensity", "proline"]
profile = wine_result.assign(cluster=labels_best).groupby("cluster")[profile_cols].mean().round(2)
print("\nCluster profiles (means):")
print(profile)

ct = pd.crosstab(cultivar_true, labels_best, rownames=["cultivar"], colnames=["cluster"])
ct.index = [cultivar_names[i] for i in ct.index]
print("\nCultivar vs cluster crosstab:")
print(ct)



---
## 9. Visualizing Clusters

### 2D scatter — two interpretable chemistry features

**Alcohol** vs **proline** — both are meaningful to domain experts and separate groups visually.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, color_vals, title in [
    (axes[0], labels_best, "K-Means clusters"),
    (axes[1], cultivar_true, "True cultivar (hidden during fit)"),
]:
    sc = ax.scatter(
        wine_df["alcohol"], wine_df["proline"],
        c=color_vals, cmap="viridis", alpha=0.75, edgecolors="k", linewidth=0.3,
    )
    ax.set_xlabel("alcohol")
    ax.set_ylabel("proline")
    ax.set_title(title)
    plt.colorbar(sc, ax=ax, label="label")

plt.tight_layout()
plt.show()



### PCA — project all 13 features to 2D for plotting

**PCA** finds directions of maximum variance. Used here **only to visualize** — not as the main clustering step.



In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_wine_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_best, cmap="viridis", alpha=0.75, edgecolors="k", linewidth=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"PCA View — K-Means K={best_k}")
plt.colorbar(label="cluster")
plt.tight_layout()
plt.show()

print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))



---
## 10. Density-Based Clustering — `DBSCAN` (Brief)

**DBSCAN** finds dense regions and marks sparse points as **noise** (`-1`).

| Parameter | Role |
|-----------|------|
| `eps` | Neighborhood radius (on scaled data) |
| `min_samples` | Minimum points to form a dense region |

**Pros**: No fixed K; can find irregular shapes  
**Cons**: Sensitive to `eps` and scaling — in 13D, many points can be labeled **noise** until `eps` is tuned.

> On Wine, try a few `eps` values on **scaled** data and inspect noise counts before trusting DBSCAN.



In [ ]:
db = DBSCAN(eps=2.2, min_samples=3)
db_labels = db.fit_predict(X_wine_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

print("DBSCAN clusters found:", n_clusters_db)
print("Noise points:", n_noise)
print("Label counts:\n", pd.Series(db_labels).value_counts().sort_index())

if n_clusters_db >= 2:
    mask = db_labels != -1
    print("Silhouette (non-noise):", round(silhouette_score(X_wine_scaled[mask], db_labels[mask]), 3))
    print("ARI vs cultivar:", round(adjusted_rand_score(cultivar_true[mask], db_labels[mask]), 3))



---
## 11. Clustering Workflow and When to Use What

```
1. Define business question (segments? product groups?)
2. Select numeric features (encode categoricals if needed)
3. Scale features (StandardScaler for K-Means)
4. Try K-Means with several K — elbow + silhouette
5. Profile clusters (means, plots, domain names)
6. Optional: compare to hidden labels if available
7. Document limitations and next steps
```

| Algorithm | Best when |
|-----------|-----------|
| **K-Means** | Roughly spherical groups; you can choose K |
| **DBSCAN** | Irregular shapes; noise/outliers matter |

### Why Wine instead of Penguins here?

| Penguins (Sessions 15–21) | Wine (today) |
|---------------------------|--------------|
| Supervised species prediction | Unsupervised grouping story |
| Students expect 3 species | Fresh domain — chemistry segments |
| Mixed numeric + categorical | All numeric — ideal for K-Means intro |

> **Link to supervised learning**: Cluster IDs can become **features** later — always validate on held-out data.



In [ ]:
labels_k3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_wine_scaled)

summary = pd.DataFrame({
    "method": ["K-Means (K=3)", f"K-Means (K={best_k})", "DBSCAN"],
    "n_groups": [3, best_k, n_clusters_db],
    "silhouette": [
        round(silhouette_score(X_wine_scaled, labels_k3), 3),
        round(silhouette_score(X_wine_scaled, labels_best), 3),
        round(silhouette_score(X_wine_scaled[db_labels != -1], db_labels[db_labels != -1]), 3) if n_clusters_db >= 2 else np.nan,
    ],
    "ARI_vs_cultivar": [
        round(adjusted_rand_score(cultivar_true, labels_k3), 3),
        round(adjusted_rand_score(cultivar_true, labels_best), 3),
        round(adjusted_rand_score(cultivar_true[db_labels != -1], db_labels[db_labels != -1]), 3) if n_clusters_db >= 2 else np.nan,
    ],
})
print(summary.to_string(index=False))



---
## 🧪 Practice Exercises

1. Run K-Means with `n_clusters=2` on wine. How do cluster profiles and the cultivar crosstab change?
2. Cluster using **only** `alcohol` and `color_intensity` (scale both). Is silhouette higher or lower than using all 13 features?
3. Try `KMeans(n_clusters=5)` — what happens to silhouette and interpretability?
4. Plot the elbow curve for the blob toy data (`X_blob`). Is K=3 obvious?
5. Tune DBSCAN: try `eps` in `[1.6, 2.0, 2.2, 2.5]` on scaled wine — how many clusters and noise points?
6. Which feature has the largest difference between cluster means for your best K? (Hint: `groupby("cluster").mean()`)
7. Write **segment names** for each cluster based on profiles (e.g. "high tannin, high alcohol").
8. Memo (6 bullets): business question, features used, K chosen, best silhouette, profile summary, deployment caution.

---

## 📝 Session 22 Summary

| Topic | Key takeaway |
|-------|--------------|
| **Unsupervised learning** | Find structure without a target column |
| **Wine dataset** | 178 rows, 13 numeric chemistry features — clustering-friendly |
| **K-Means** | Partition into K groups by distance to centroids |
| **Scaling** | Required when features use different units (`proline` vs `hue`) |
| **Choosing K** | Elbow (inertia) + silhouette score |
| **Evaluation** | Profiles first; silhouette without labels; ARI only when labels exist |
| **DBSCAN** | Density-based; finds noise; no fixed K |

### Key gotchas
- **Cluster IDs are arbitrary** — profile and name each segment
- **K-Means assumes roughly spherical clusters** — poor on elongated or nested shapes
- **Do not pass labels into `fit`** — cultivar is for interpretation only
- **Silhouette and ARI can disagree** — use multiple views
- **Wine is small (n=178)** — treat metrics as illustrative, not production guarantees

### Next Session
**Session 23**: *(your choice — e.g. full ML capstone project, PCA for dimensionality reduction, or end-of-course review and portfolio wrap-up.)*

